# BM25 + география + признаки объявления

Проверяем, помогают ли контактные флаги, рейтинг, число отзывов и совпадение категории переранжировать один и тот же пул BM25-кандидатов. База — `BM25/max(BM25) + 0.2·same_location + 0.2·exp(−distance/100 км)`. Признаки не меняют весь список: они могут переставить только первые `R` объявлений базовой выдачи (`R ∈ {50, 100, 300, 1000}`), так что дальний кандидат не перепрыгнет через сильные результаты только из-за рейтинга.

Подбираем вес и глубину rerank на 500 val-контекстах; проверяем на 5000 test-контекстах (разбиение запросов `seed=42`, выборки `seed=43`). Корпус объявлений — весь train; метки отложенных запросов используются только для Recall@50. Центры локаций оцениваются на train-части. Предыдущий геоэксперимент уже использовал test для оценки базовой формулы, поэтому результат исследовательский.

`search_category` можно сопоставить с `item_category_id`, но у запроса нет `item_microcat_id`/подкатегории. Совпадение категории используется только как мягкий late-rerank сигнал; неизвестная категория не штрафуется. Контактность и репутация — предпочтения/качество объявления, а не текстовая релевантность. Их эффект оцениваем отдельно и вместе; положительный вес автоматически выбирается только если помогает на val.

## Настройки и данные

In [ ]:
from pathlib import Path
import sys
import gc

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/split.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.split import split_queries
from src.retrievers.bm25 import BM25Retriever

SPLIT_SEED = 42
SAMPLE_SEED = 43
N_VAL_QUERIES = 500
N_TEST_QUERIES = 5000
N_CANDIDATES = 3000
RERANK_DEPTHS = [50, 100, 300, 1000]
K = 50
WEIGHTS = [-0.2, -0.1, -0.05, 0.0, 0.05, 0.1, 0.2]
QUERY_COLS = ["search_query", "search_location_id", "search_is_delivery_search", "search_infm_params_text", "search_category"]
ITEM_COLS = ["item_id", "item_title_raw", "item_location_id", "item_latitude", "item_longitude",
             "item_is_phone_hidden", "item_is_message_forbidden", "item_rating", "item_rating_reviews_count",
             "item_category_id", "item_microcat_id"]
SIGNALS = ["phone_hidden", "message_forbidden", "rating_gap", "review_count", "category_mismatch"]

In [ ]:
data = pl.read_parquet(ROOT / "data/train.parquet", columns=list(dict.fromkeys(QUERY_COLS + ITEM_COLS)))
data = data.with_columns(
    pl.Series("split", split_queries(data["search_query"], seed=SPLIT_SEED)),
    pl.col("item_latitude", "item_longitude").cast(pl.Float64),
)
items = pd.DataFrame(
    data.select(ITEM_COLS).unique(subset="item_id", keep="last", maintain_order=True)
    .sort("item_id").to_dict(as_series=False)
)
train_items = data.filter(pl.col("split") == "train").unique(subset="item_id", keep="last", maintain_order=True)
centers = (
    train_items.filter(pl.col("item_latitude").is_between(-90, 90) & pl.col("item_longitude").is_between(-180, 180))
    .group_by("item_location_id")
    .agg(pl.col("item_latitude").median().alias("center_lat"),
         pl.col("item_longitude").median().alias("center_lon"))
)
centers = pd.DataFrame(centers.to_dict(as_series=False)).set_index("item_location_id")
queries, targets = {}, {}
for split in ("val", "test"):
    part = (
        data.filter(pl.col("split") == split)
        .group_by(QUERY_COLS).agg(pl.col("item_id").unique().alias("targets"))
        .sort(QUERY_COLS)
        .sample(n=N_VAL_QUERIES if split == "val" else N_TEST_QUERIES,
                seed=SAMPLE_SEED, shuffle=True)
    )
    queries[split] = pd.DataFrame(part.select(QUERY_COLS).to_dict(as_series=False)).join(centers, on="search_location_id")
    targets[split] = [set(ids) for ids in part["targets"].to_list()]
    assert all(targets[split])
    assert set().union(*targets[split]).issubset(set(items.item_id))
    print(split, "контекстов:", len(part), "без центра:", queries[split].center_lat.isna().sum())
print("Объявлений в корпусе:", len(items), "центров:", len(centers))
print("Частые пары категория запроса / категория объявления:")
display(data.group_by("search_category", "item_category_id").len().sort("len", descending=True).head(12))
print("Подкатегорий в объявлениях:", items.item_microcat_id.nunique(), "— соответствующего query-поля в этом датасете нет")
del data, train_items
gc.collect()

## Один BM25-индекс и признаки кандидатов

Флаги означают **отсутствие** канала связи. `rating_gap = (5 − rating)/5`, пропуски и оценки 5 дают 0. `review_count = log1p(count)/log1p(p99)`, ограничено `[0, 1]`; p99 вычисляется по корпусу без использования меток. Совпадение макрокатегории штрафуется только при известной категории запроса. Длина пула и score BM25 одинаковы для всех вариантов. Признаки применяются только внутри top-R базового ранжирования.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

review_scale = np.log1p(items["item_rating_reviews_count"].fillna(0).clip(lower=0).quantile(0.99))
assert review_scale > 0
metadata = items.set_index("item_id")
model = BM25Retriever(lemmatize=False, batch_size=16).fit(items[["item_id", "item_title_raw"]])
features = {}
for split in ("val", "test"):
    q = queries[split]
    shape = (len(q), N_CANDIDATES)
    f = {
        "base": np.full(shape, -np.inf, dtype=np.float32),
        "hit": np.zeros(shape, dtype=bool),
        "denom": np.array([len(t) for t in targets[split]]),
    }
    for signal in SIGNALS:
        f[signal] = np.zeros(shape, dtype=np.float32)
    for start in range(0, len(q), 32):
        candidates = model.retrieve(q.iloc[start:start + 32], k=N_CANDIDATES)
        for local_i, group in candidates.groupby("query_index", sort=False):
            i = start + int(local_i)
            group = group.sort_values("rank")
            ids = group.item_id.to_numpy()
            count = len(ids)
            item = metadata.loc[ids]
            scores = group.score.to_numpy()
            same = (item.item_location_id.to_numpy() == q.iloc[i].search_location_id).astype(np.float32)
            distance = haversine(q.iloc[i].center_lat, q.iloc[i].center_lon,
                                 item.item_latitude.to_numpy(), item.item_longitude.to_numpy())
            valid = (np.isfinite(distance) & item.item_latitude.between(-90, 90).to_numpy()
                     & item.item_longitude.between(-180, 180).to_numpy())
            geo = np.exp(-np.where(valid, distance, np.inf) / 100)
            f["base"][i, :count] = scores / scores.max() + 0.2 * same + 0.2 * geo
            f["phone_hidden"][i, :count] = item.item_is_phone_hidden.to_numpy(dtype=np.float32)
            f["message_forbidden"][i, :count] = item.item_is_message_forbidden.to_numpy(dtype=np.float32)
            f["rating_gap"][i, :count] = (5 - item.item_rating.fillna(5).clip(0, 5)).to_numpy(dtype=np.float32) / 5
            counts = item.item_rating_reviews_count.fillna(0).clip(lower=0).to_numpy(dtype=np.float32)
            f["review_count"][i, :count] = np.clip(np.log1p(counts) / review_scale, 0, 1)
            query_category = q.iloc[i].search_category
            item_categories = item.item_category_id.to_numpy()
            if pd.notna(query_category) and query_category != 0:
                f["category_mismatch"][i, :count] = (pd.isna(item_categories) | (item_categories != query_category)).astype(np.float32)
            f["hit"][i, :count] = [item_id in targets[split][i] for item_id in ids]
    features[split] = f
    print(split, "готово; без кандидатов:", (~np.isfinite(f["base"]).any(axis=1)).sum(),
          "Recall@3000:", (f["hit"].sum(axis=1) / f["denom"]).mean())
del model, metadata, candidates
gc.collect()

## Подбор отдельных весов и глубины на val

Для каждой фичи выбираем на val пару `(R, вес)`. Сначала выдача сортируется по базовому BM25+гео score; сигнал может переставить кандидатов только внутри первых R позиций. При равном Recall предпочитаем меньший R и меньший модуль веса.

In [ ]:
def recall_per_query(f, weights, depth=0):
    # Сначала фиксируем порядок базы. Utility сигнал действует только в префиксе.
    base_order = np.argsort(-f["base"], axis=1, kind="stable")
    order = base_order.copy()
    if depth:
        prefix = base_order[:, :depth]
        rows = np.arange(len(prefix))[:, None]
        utility = np.zeros(prefix.shape, dtype=np.float32)
        for signal, weight in weights.items():
            if weight:
                utility += weight * f[signal][rows, prefix]
        local_order = np.argsort(-(f["base"][rows, prefix] + utility), axis=1, kind="stable")
        order[:, :depth] = np.take_along_axis(prefix, local_order, axis=1)
    top = order[:, :K]
    return np.take_along_axis(f["hit"], top, axis=1).sum(axis=1) / f["denom"]

baseline = recall_per_query(features["val"], {})
selected_weights, selected_depths = {}, {}
validation = []
for signal in SIGNALS:
    runs = [(depth, weight, recall_per_query(features["val"], {signal: weight}, depth).mean())
            for depth in RERANK_DEPTHS for weight in WEIGHTS]
    depth, weight, score = max(runs, key=lambda row: (row[2], -row[0], -abs(row[1])))
    selected_weights[signal], selected_depths[signal] = weight, depth
    validation.append({"variant": signal, "rerank_depth": depth, "weight": weight,
                       "val_recall": score, "delta_val": score - baseline.mean()})
validation = pd.DataFrame(validation)
display(validation)
print("Base val Recall@50:", baseline.mean())
joint_depth = max(selected_depths.values())
print("Joint val Recall@50:", recall_per_query(features["val"], selected_weights, joint_depth).mean(),
      "(depth", joint_depth, ")")

## Проверка на test

Проверяем выбранные на val веса и глубины на фиксированном test. Парный bootstrap по запросам показывает неопределённость разницы Recall; интервал, содержащий 0, не даёт уверенного вывода о приросте.

In [ ]:
test_base = recall_per_query(features["test"], {})
rng = np.random.default_rng(123)
bootstrap_indices = rng.integers(0, len(test_base), size=(2000, len(test_base)))
rows = []
variants = [("base", {}, 0)]
variants += [(s, {s: selected_weights[s]}, selected_depths[s]) for s in SIGNALS]
variants += [("joint", selected_weights, joint_depth)]
for variant, weights, depth in variants:
    recall = recall_per_query(features["test"], weights, depth)
    delta = recall - test_base
    low, high = np.quantile(delta[bootstrap_indices].mean(axis=1), [0.025, 0.975])
    rows.append({"variant": variant, "rerank_depth": depth,
                 "weight": selected_weights.get(variant, np.nan),
                 "test_recall": recall.mean(), "delta_test": delta.mean(),
                 "delta_95%_low": low, "delta_95%_high": high})
comparison = pd.DataFrame(rows)
display(comparison)
print("Пул остаётся BM25 top-3000; признаки меняют только порядок внутри выбранного top-R.")